Absolutely. You only need the **questions**, without answers or code.

## SQL + PySpark Interview Coding Questions

## Additional 3–5 YOE Interview Questions

### Q9. Top 3 Salaries in Each Department

Write a query to find the **top 3 highest distinct salaries in each department**.

---

### Q10. Third Highest Salary

Write a query to find the **third highest distinct salary** from the Employee table.

---

### Q11. Employees Sharing the Same Salary

Find all employees whose salary is **shared by one or more other employees**.

---

### Q12. Departments with More Than 2 Employees

Find all departments having **more than 2 employees**.

---

### Q13. Employees Without Managers

Find all employees who **do not have a manager assigned**.

---

### Q14. Customers Who Never Placed an Order

Find customers who **have never placed any order**.

---

### Q15. Orders Above Customer's Average Order Amount

Find orders where the **order amount is greater than that customer's average order amount**.

---

### Q16. First Order of Every Customer

Find the **first order placed by every customer**.

---

### Q17. Second Order of Every Customer

Find the **second order placed by every customer**.

---

### Q18. Latest Order Amount Greater Than Previous Order

Find customers whose **latest order amount is greater than their previous order amount**.

---

### Q19. Compare Current Salary with Previous Salary

Given employee salary history, compare each employee's **current salary with their previous salary**.

---

### Q20. Consecutive Order Days

Find customers who placed orders on **consecutive days**.

---

### Q21. Find Missing Dates

Given a table containing dates, identify the **dates missing from the expected date range**.

---

### Q22. Gaps and Islands

Given employee attendance/order activity data, identify **continuous periods (streaks) of activity**.

---

### Q23. Month-over-Month Sales Growth

Calculate the **month-over-month percentage growth in sales**.

---

### Q24. Year-over-Year Sales Growth

Calculate the **year-over-year percentage growth in sales**.

---

### Q25. Percentage Contribution to Total Salary

For each employee, calculate what **percentage of the total salary paid by the company** their salary represents.

---

### Q26. Highest Salary Employee per Department

Find the employee(s) having the **highest salary in each department**.

---

### Q27. Employees Above Department Average

Find employees whose salary is **greater than the average salary of their department**.

---

### Q28. Customers with Orders in Consecutive Months

Find customers who placed orders in **two or more consecutive months**.

---

### Q29. Source vs Target Change Detection

Given source and target tables, identify records that are:

* **INSERT**
* **UPDATE**
* **DELETE**
* **NO CHANGE**

---

### Q30. SCD Type 2 Implementation

Implement an **SCD Type 2 solution using Delta Lake MERGE** to maintain historical employee/customer records.

---

### Q31. Records Present in Source but Missing in Target

Compare source and target tables and find records that **exist in the source but not in the target**.

---

### Q32. Records Present in Target but Missing in Source

Compare source and target tables and find records that **exist in the target but not in the source**.

---

### Q33. Full Source vs Target Data Comparison

Compare source and target tables and identify **new, deleted, changed, and unchanged records**.

---

### Q34. Deduplicate Business Records

Given multiple records for the same business key, **deduplicate the data and keep the latest record based on timestamp**.

---

### Q35. NULL vs Blank Values

Find records where a column contains either **NULL values or blank/empty strings** and handle them appropriately.

---

### Q36. Pivot Monthly Sales

Given sales data containing `year`, `month`, and `sales`, create a **pivot showing each month as a separate column**.

---

### Q37. Unpivot Data

Given a table containing monthly columns such as `Jan`, `Feb`, `Mar`, etc., convert it into a **row-based/unpivoted format**.

---

### Q38. Flatten Nested Arrays

Given customer/order data containing an **array column**, flatten the array using Spark functions such as `explode`.

---

### Q39. Parse and Flatten JSON

Given a column containing **JSON strings with nested structures and arrays**, parse the JSON and flatten it into a relational DataFrame.

---

### Q40. Duplicate Detection and Performance

Given a large dataset containing duplicate records:

1. How would you identify duplicates?
2. How would you remove them?
3. How would you optimize the operation in **Spark/Databricks**?
4. What happens during the **shuffle**?
5. How would you handle **data skew**?
6. Which Spark functions/optimizations would you consider?


# SQL + PySpark Interview Coding Practice — Questions 9 to 40

**Target:** 3–5 YOE Data Engineer / Databricks Engineer

Run the notebook from top to bottom. Each question has executable SQL and PySpark.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

employee_df = spark.createDataFrame([
    (1,"John",10,10000,None),(2,"Alice",10,15000,1),(3,"Bob",10,12000,1),
    (4,"Carol",20,20000,2),(5,"David",20,18000,2),(6,"Eva",20,18000,2),
    (7,"Frank",30,9000,None),(8,"Grace",30,11000,7),(9,"Henry",30,13000,7)
], ["emp_id","emp_name","dept_id","salary","manager_id"])

customers_df = spark.createDataFrame(
    [(101,"A"),(102,"B"),(103,"C"),(104,"D")],
    ["customer_id","customer_name"]
)

orders_df = spark.createDataFrame([
    (1,101,"2023-01-05",100),(2,101,"2023-02-10",150),
    (3,101,"2023-03-15",200),(4,102,"2023-01-12",500),
    (5,102,"2023-03-18",300),(6,103,"2023-01-20",50),
    (7,103,"2023-02-20",75),(8,103,"2023-03-20",100),
    (9,104,"2023-03-25",1000),(10,101,"2023-03-25",250)
], ["order_id","customer_id","order_date","amount"])

sales_df = spark.createDataFrame([
    ("2023-01",1000),("2023-02",1500),("2023-03",2000),
    ("2023-04",1200),("2023-05",2500)
], ["month","sales"])

employee_df.createOrReplaceTempView("employee")
customers_df.createOrReplaceTempView("customers")
orders_df.createOrReplaceTempView("orders")
sales_df.createOrReplaceTempView("sales")

## Q9 — Top 3 salaries in each department

In [0]:
%sql
SELECT * FROM (
SELECT *, DENSE_RANK() OVER(PARTITION BY dept_id ORDER BY salary DESC) rnk
FROM employee) WHERE rnk <= 3 ORDER BY dept_id,salary DESC;

In [0]:
w = Window.partitionBy("dept_id").orderBy(F.col("salary").desc())
employee_df.withColumn("rnk",F.dense_rank().over(w)).filter("rnk <= 3").show()

## Q10 — 3rd highest distinct salary

In [0]:
%sql
SELECT salary FROM (
SELECT DISTINCT salary,DENSE_RANK() OVER(ORDER BY salary DESC) rnk FROM employee
) WHERE rnk=3;

In [0]:
employee_df.select("salary").distinct().withColumn(
    "rnk",F.dense_rank().over(Window.orderBy(F.col("salary").desc()))
).filter("rnk=3").show()

## Q11 — Salaries shared by multiple employees

In [0]:
%sql
SELECT salary,COUNT(*) employee_count FROM employee
GROUP BY salary HAVING COUNT(*)>1 ORDER BY salary DESC;

In [0]:
employee_df.groupBy("salary").count().filter("count>1").orderBy(F.col("salary").desc()).show()

## Q12 — Departments with more than 2 employees

In [0]:
%sql
SELECT dept_id,COUNT(*) employee_count FROM employee
GROUP BY dept_id HAVING COUNT(*)>2;

In [0]:
employee_df.groupBy("dept_id").count().filter("count>2").show()

## Q13 — Employees without a manager

In [0]:
%sql
SELECT * FROM employee WHERE manager_id IS NULL;

In [0]:
employee_df.filter(F.col("manager_id").isNull()).show()

## Q14 — Customers who never placed an order

In [0]:
%sql
SELECT c.* FROM customers c LEFT JOIN orders o
ON c.customer_id=o.customer_id WHERE o.customer_id IS NULL;

In [0]:
customers_df.join(
    orders_df.select("customer_id").distinct(),
    "customer_id","left_anti"
).show()

## Q15 — Orders above customer's average

In [0]:
%sql
SELECT * FROM (
SELECT *,AVG(amount) OVER(PARTITION BY customer_id) customer_avg
FROM orders) WHERE amount>customer_avg;

In [0]:
orders_df.withColumn(
    "customer_avg",F.avg("amount").over(Window.partitionBy("customer_id"))
).filter(F.col("amount")>F.col("customer_avg")).show()

## Q16 — First order of every customer

In [0]:
w_order = Window.partitionBy("customer_id").orderBy(
    F.to_date("order_date"),"order_id"
)

In [0]:
%sql
SELECT * FROM (
SELECT *,ROW_NUMBER() OVER(
PARTITION BY customer_id ORDER BY CAST(order_date AS DATE),order_id) rn
FROM orders) WHERE rn=1;

In [0]:
orders_df.withColumn("rn",F.row_number().over(w_order)).filter("rn=1").drop("rn").show()

## Q17 — Second order of every customer

In [0]:
orders_df.withColumn("rn",F.row_number().over(w_order)).filter("rn=2").drop("rn").show()

## Q18 — Latest order amount greater than previous order

In [0]:
%sql
WITH x AS (
SELECT *,
LAG(amount) OVER(PARTITION BY customer_id ORDER BY CAST(order_date AS DATE),order_id) previous_amount,
ROW_NUMBER() OVER(PARTITION BY customer_id ORDER BY CAST(order_date AS DATE) DESC,order_id DESC) latest_rn
FROM orders)
SELECT * FROM x WHERE latest_rn=1 AND amount>previous_amount;

In [0]:
w_asc = Window.partitionBy("customer_id").orderBy(
    F.to_date("order_date"),"order_id"
)
w_desc = Window.partitionBy("customer_id").orderBy(
    F.to_date("order_date").desc(),"order_id"
)

orders_df.withColumn(
    "previous_amount",F.lag("amount").over(w_asc)
).withColumn(
    "latest_rn",F.row_number().over(w_desc)
).filter(
    (F.col("latest_rn")==1)&(F.col("amount")>F.col("previous_amount"))
).show()

## Q19 — Compare current salary with previous salary

In [0]:
%sql
SELECT emp_id,emp_name,salary,LAG(salary) OVER(ORDER BY salary) previous_salary
FROM employee;

In [0]:
employee_df.withColumn(
    "previous_salary",F.lag("salary").over(Window.orderBy("salary"))
).show()

## Q20 — Consecutive order days

In [0]:
dates_df = orders_df.select(
    "customer_id",F.to_date("order_date").alias("order_date")
).distinct()

In [0]:
%sql
WITH d AS (
SELECT DISTINCT customer_id,CAST(order_date AS DATE) order_date FROM orders),
x AS (
SELECT *,LAG(order_date) OVER(
PARTITION BY customer_id ORDER BY order_date) previous_date FROM d)
SELECT * FROM x WHERE DATEDIFF(order_date,previous_date)=1;

In [0]:
dates_df.withColumn(
    "previous_date",
    F.lag("order_date").over(Window.partitionBy("customer_id").orderBy("order_date"))
).filter(
    F.datediff("order_date","previous_date")==1
).show()

## Q21 — Find missing dates

In [0]:
calendar_df = spark.sql("""
SELECT explode(
  sequence(to_date('2023-01-01'),to_date('2023-03-31'),interval 1 day)
) AS calendar_date
""")

order_dates_df = orders_df.select(
    F.to_date("order_date").alias("order_date")
).distinct()

calendar_df.join(
    order_dates_df,
    calendar_df.calendar_date==order_dates_df.order_date,
    "left_anti"
).show()

## Q22 — Gaps and islands: consecutive streaks

In [0]:
numbered = dates_df.withColumn(
    "rn",F.row_number().over(
        Window.partitionBy("customer_id").orderBy("order_date")
    )
)

islands = numbered.withColumn(
    "grp",F.date_sub("order_date",F.col("rn").cast("int"))
)

islands.groupBy("customer_id","grp").agg(
    F.min("order_date").alias("start_date"),
    F.max("order_date").alias("end_date"),
    F.count("*").alias("streak_length")
).orderBy(
    "customer_id",F.col("streak_length").desc()
).show()

## Q23 — Month-over-month sales growth

In [0]:
%sql
WITH x AS (
SELECT month,sales,LAG(sales) OVER(ORDER BY month) previous_sales
FROM sales)
SELECT month,sales,previous_sales,
ROUND((sales-previous_sales)/previous_sales*100,2) growth_percentage
FROM x;

In [0]:
sales_df.withColumn(
    "previous_sales",F.lag("sales").over(Window.orderBy("month"))
).withColumn(
    "growth_percentage",
    F.round(
        (F.col("sales")-F.col("previous_sales"))
        /F.col("previous_sales")*100,2
    )
).show()

## Q24 — Year-over-year growth

In [0]:
year_sales_df = spark.createDataFrame(
    [(2022,10000),(2023,12500),(2024,15000)],
    ["year","sales"]
)

In [0]:
year_sales_df.withColumn(
    "previous_year_sales",
    F.lag("sales").over(Window.orderBy("year"))
).withColumn(
    "yoy_growth",
    F.round(
        (F.col("sales")-F.col("previous_year_sales"))
        /F.col("previous_year_sales")*100,2
    )
).show()

## Q25 — Percentage contribution to total salary

In [0]:
%sql
SELECT emp_id,emp_name,salary,
ROUND(salary/SUM(salary) OVER()*100,2) salary_percentage
FROM employee;

In [0]:
employee_df.withColumn(
    "salary_percentage",
    F.round(
        F.col("salary")/
        F.sum("salary").over(Window.partitionBy())*100,2
    )
).show()

## Q26 — Highest salary employee in each department

In [0]:
%sql
SELECT * FROM (
SELECT *,ROW_NUMBER() OVER(
PARTITION BY dept_id ORDER BY salary DESC,emp_id) rn
FROM employee) WHERE rn=1;

In [0]:
employee_df.withColumn(
    "rn",F.row_number().over(
        Window.partitionBy("dept_id").orderBy(
            F.col("salary").desc(),"emp_id"
        )
    )
).filter("rn=1").drop("rn").show()

## Q27 — Employees above department average

In [0]:
%sql
SELECT * FROM (
SELECT *,AVG(salary) OVER(PARTITION BY dept_id) dept_avg_salary
FROM employee) WHERE salary>dept_avg_salary;

In [0]:
employee_df.withColumn(
    "dept_avg_salary",
    F.avg("salary").over(Window.partitionBy("dept_id"))
).filter(
    F.col("salary")>F.col("dept_avg_salary")
).show()

## Q28 — Customers with orders in consecutive months

In [0]:
months_df = orders_df.select(
    "customer_id",
    F.trunc(F.to_date("order_date"),"month").alias("order_month")
).distinct()

months_df.withColumn(
    "previous_month",
    F.lag("order_month").over(
        Window.partitionBy("customer_id").orderBy("order_month")
    )
).filter(
    F.months_between("order_month","previous_month")==1
).show()

## Q29 — Source vs target: INSERT / UPDATE / DELETE / NO CHANGE

In [0]:
source_df = spark.createDataFrame(
    [(1,"A","h1"),(2,"B_new","h2_new"),(4,"D","h4")],
    ["customer_id","customer_name","hash_value"]
)

target_df = spark.createDataFrame(
    [(1,"A","h1",True),(2,"B","h2",True),(3,"C","h3",True)],
    ["customer_id","customer_name","hash_value","is_current"]
)

source_df.createOrReplaceTempView("source_customer")
target_df.createOrReplaceTempView("target_customer")

In [0]:
%sql
SELECT COALESCE(s.customer_id,t.customer_id) customer_id,
CASE
WHEN s.customer_id IS NULL THEN 'DELETE'
WHEN t.customer_id IS NULL THEN 'INSERT'
WHEN s.hash_value<>t.hash_value THEN 'UPDATE'
ELSE 'NO CHANGE' END change_type
FROM source_customer s
FULL OUTER JOIN target_customer t
ON s.customer_id=t.customer_id;

## Q30 — SCD Type 2 Delta MERGE pattern

This is the common interview pattern. In production, stage the source so the
expired version and new current version are both handled correctly.

```sql
MERGE INTO target t
USING source s
ON t.customer_id=s.customer_id AND t.is_current=true
WHEN MATCHED AND t.hash_value<>s.hash_value THEN
  UPDATE SET t.is_current=false,t.end_date=current_date()
WHEN NOT MATCHED THEN
  INSERT (customer_id,customer_name,hash_value,start_date,end_date,is_current)
  VALUES (s.customer_id,s.customer_name,s.hash_value,current_date(),NULL,true);
```

## Q31 — Source records missing from target

In [0]:
source_df.join(
    target_df.select("customer_id").distinct(),
    "customer_id","left_anti"
).show()

## Q32 — Target records missing from source

In [0]:
target_df.join(
    source_df.select("customer_id").distinct(),
    "customer_id","left_anti"
).show()

## Q33 — Full source/target change detection

In [0]:
%sql
SELECT COALESCE(s.customer_id,t.customer_id) customer_id,
CASE WHEN s.customer_id IS NULL THEN 'DELETE'
WHEN t.customer_id IS NULL THEN 'INSERT'
WHEN s.hash_value<>t.hash_value THEN 'UPDATE'
ELSE 'NO CHANGE' END change_type
FROM source_customer s FULL OUTER JOIN target_customer t
ON s.customer_id=t.customer_id;

## Q34 — Deduplicate and keep latest business record

In [0]:
orders_df.withColumn(
    "rn",F.row_number().over(
        Window.partitionBy("customer_id").orderBy(
            F.to_date("order_date").desc(),"order_id"
        )
    )
).filter("rn=1").drop("rn").show()

## Q35 — NULL vs blank values

In [0]:
null_df = spark.createDataFrame(
    [(1,None),(2,""),(3,"   "),(4,"ABC")],
    ["id","value"]
)

null_df.filter(
    F.col("value").isNull() | (F.trim("value")=="")
).show()

## Q36 — Pivot monthly sales

In [0]:
sales_df.groupBy().pivot("month").sum("sales").show()

## Q37 — Unpivot using Spark SQL stack()

In [0]:
quarter_df = spark.createDataFrame(
    [(1,100,200,300)],
    ["id","q1","q2","q3"]
)
quarter_df.createOrReplaceTempView("quarter_sales")

In [0]:
%sql
SELECT id,quarter,sales
FROM quarter_sales
LATERAL VIEW stack(3,'q1',q1,'q2',q2,'q3',q3) s AS quarter,sales;

## Q38 — Flatten arrays using explode()

In [0]:
array_df = spark.createDataFrame(
    [(1,["A","B","C"]),(2,["D","E"])],
    ["id","items"]
)
array_df.withColumn("item",F.explode("items")).show()

## Q39 — Parse and flatten JSON

In [0]:
json_df = spark.createDataFrame([
    ('{"id":1,"name":"John","skills":["Spark","SQL"]}',),
    ('{"id":2,"name":"Alice","skills":["Python","Databricks"]}',)
],["json_string"])

parsed = json_df.withColumn(
    "data",
    F.from_json("json_string","id INT,name STRING,skills ARRAY<STRING>")
)

parsed.select(
    "data.id","data.name",F.explode("data.skills").alias("skill")
).show()

## Q40 — Find duplicates efficiently on a large dataset

In [0]:
duplicate_check = (
    orders_df
    .groupBy("customer_id","order_date")
    .count()
    .filter(F.col("count")>1)
)

duplicate_check.show()

### Performance follow-up

Discuss:
- shuffle from groupBy/window/join
- data skew
- broadcast joins
- AQE
- partition pruning
- small files
- Delta OPTIMIZE/clustering
- `explain("formatted")`

SQL + PYSPARK + DATABRICKS INTERVIEW THEORY
===============================================

Target: 3–5 YOE Data Engineer / Databricks Engineer

1. SQL FUNDAMENTALS
-------------------
WHERE vs HAVING:
WHERE filters rows before aggregation. HAVING filters groups after GROUP BY.

GROUP BY vs Window:
GROUP BY reduces rows. A window function calculates across related rows while
preserving row-level detail.

UNION vs UNION ALL:
UNION removes duplicates. UNION ALL preserves them and generally avoids the
deduplication work.

INNER JOIN vs LEFT JOIN:
INNER returns matching rows. LEFT returns all left rows plus matching right rows.

LEFT ANTI JOIN:
Rows in the left dataset with no match in the right dataset.
Very useful for source-minus-target validation.

LEFT SEMI JOIN:
Left rows for which a match exists on the right, without returning right columns.

2. WINDOW FUNCTIONS
-------------------
ROW_NUMBER:
Every row gets a unique sequence. Use when exactly one row must be selected.

RANK:
Ties share a rank; gaps can occur.

DENSE_RANK:
Ties share a rank; no gaps. Good for Nth highest DISTINCT values.

PARTITION BY:
Creates independent window groups.

LAG:
Reads a previous row. Useful for previous transaction, MoM growth and change detection.

LEAD:
Reads a following row.

ROWS vs RANGE:
ROWS is based on row positions. RANGE is based on ORDER BY values and peer values.
Be explicit for cumulative calculations when ordering values can repeat.

3. NULL HANDLING
----------------
`column = NULL` is incorrect.
Use IS NULL / IS NOT NULL.

COALESCE returns the first non-NULL expression.

NULL and empty string are different:
NULL = missing/unknown.
'' = real zero-length string.
TRIM can identify whitespace-only values.

4. DEDUPLICATION
----------------
Latest-record pattern:
ROW_NUMBER() OVER (
  PARTITION BY business_key
  ORDER BY event_time DESC, tie_breaker DESC
)

MAX(timestamp) alone does not return the complete corresponding row.
Use a deterministic tie-breaker when timestamps can tie.

5. DATE / TIME QUESTIONS
------------------------
Important Spark functions:
to_date, year, month, date_add, date_sub, datediff,
months_between, add_months, trunc, date_format.

Missing dates:
Generate a calendar with sequence + explode and use LEFT ANTI JOIN.

Consecutive dates:
Use LAG and date differences or gaps-and-islands.

6. GAPS AND ISLANDS
-------------------
Typical technique:
1. Sort rows.
2. Assign ROW_NUMBER.
3. Subtract row number from the date.
4. Equal keys form an island.
5. GROUP BY the island key.

Used for login streaks, consecutive orders, attendance and active periods.

7. SCD TYPE 2
-------------
SCD Type 2 preserves historical versions.

Typical columns:
business_key
attributes
start_date
end_date
is_current
hash_value

Change detection:
Match on business key and compare tracked attributes or a hash.

When current data changes:
1. Expire old version: is_current=false, end_date=current_date().
2. Insert new version: start_date=current_date(), end_date=NULL, is_current=true.

A hash can simplify comparison of many tracked attributes.

8. DELTA / DATABRICKS
---------------------
MERGE:
Conditional INSERT/UPDATE/DELETE operation for Delta tables.

Time Travel:
Query historical Delta table versions by version or timestamp.

Schema Enforcement:
Validates writes against the target schema.

Schema Evolution:
Allows supported schema changes when configured.

OPTIMIZE:
Compacts small files.

Z-ORDER:
Improves data layout/data skipping for suitable filter columns.

VACUUM:
Removes old unreferenced files after retention constraints.

9. SPARK EXECUTION / PERFORMANCE
--------------------------------
Shuffle:
Redistributes records across partitions.
Common causes: groupBy, join, distinct, orderBy, repartition and many windows.

Narrow transformation:
No broad redistribution. Examples: filter and map-style operations.

Wide transformation:
Requires redistribution. Examples: groupBy, join, distinct, repartition.

Data skew:
A small number of keys contain a disproportionate amount of data, creating
straggler tasks.

Skew handling:
- AQE skew join handling
- salting
- better partitioning
- broadcast a genuinely small side
- filter/project early
- analyze key distribution

AQE:
Adaptive Query Execution uses runtime statistics to improve execution.
Important features include post-shuffle partition coalescing, skew handling and
runtime join strategy changes.

Predicate pushdown:
Push filters closer to the source so less data is read.

Partition pruning:
Skip table partitions that cannot satisfy a partition-column filter.

Small files:
Too many files increase listing and scheduling overhead. Delta compaction/OPTIMIZE
can help.

10. SPARK JOINS
---------------
Broadcast join:
Broadcasts a small side to executors so a large-side shuffle may be avoided.
Only use when the broadcast side fits safely in memory.

Sort-merge join:
Large datasets are commonly shuffled by join key and sorted before matching.

Shuffle hash join:
Both sides are shuffled by key and hash-based matching is performed within
partitions. Spark's optimizer decides whether it is appropriate.

Inspect execution:
df.explain("formatted")
and the Spark SQL UI.

11. PYSPARK TRANSLATIONS
------------------------
SQL WHERE       -> df.filter(...)
SQL SELECT      -> df.select(...)
SQL GROUP BY    -> df.groupBy(...).agg(...)
SQL JOIN        -> df.join(...)
ROW_NUMBER      -> F.row_number().over(window)
DENSE_RANK      -> F.dense_rank().over(window)
LAG             -> F.lag(...).over(window)
EXPLODE         -> F.explode(...)

12. INTERVIEW PERFORMANCE FOLLOW-UPS
------------------------------------
After solving a coding problem, be ready for:
- Does it shuffle?
- Which operator causes the shuffle?
- Can it be optimized?
- Is the key skewed?
- Can a small side be broadcast?
- What happens at 1 TB scale?
- What happens with duplicate join keys?
- What happens with NULL join keys?
- What happens with a highly skewed window partition?
- How do you inspect the physical plan?
- How does AQE help?
- How would you handle small files?
- Would you partition the Delta table?
- What would you monitor in Spark UI?

13. RAPID-FIRE REVISION
-----------------------
WHERE vs HAVING
UNION vs UNION ALL
ROW_NUMBER vs RANK vs DENSE_RANK
GROUP BY vs Window
INNER vs LEFT JOIN
LEFT ANTI vs LEFT SEMI
COUNT(*) vs COUNT(column)
NULL vs blank
MAX(date) vs ROW_NUMBER for latest row
Shuffle
Skew
AQE
Broadcast join
Sort-merge join
Partition pruning
Predicate pushdown
Delta MERGE
SCD2
OPTIMIZE
Z-ORDER
VACUUM
Time travel
Schema enforcement
Schema evolution

14. HOW TO ANSWER A CODING QUESTION
------------------------------------
1. Clarify expected output.
2. Identify business key.
3. Identify duplicate/tie behavior.
4. Decide aggregation vs window.
5. Write SQL.
6. Explain each step.
7. Translate to PySpark.
8. Discuss edge cases.
9. Discuss Spark execution/performance.

Example:
"Find latest order per customer."

Strong explanation:
"I will partition by customer_id, order by order_date descending and use ROW_NUMBER.
I will keep rn=1. I will add order_id as a tie-breaker so the result is deterministic."

Then discuss duplicate dates, NULLs, skew and the shuffle caused by the window.


In [0]:
#input
id  category
171  tata
null maruthi
null hyundai
123  MG hector
null  kia 
null  ford 

#output 
id  category
171  tata
171 maruthi
171 hyundai
123  MG hector
123  kia 
123  ford 


In [0]:
name 
virat kohli
sachin ramesh tendulkar 
dhoni 

first_name  middle_name last_name
 virat        null       kohli 
 sachin       ramesh     tendulkar 
 dhoni        null       null